# 🧮 Naive Bayes
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Naive Bayes extends the Bayesian classification idea with one strong simplification: assume features are conditionally independent given the class. Despite this "naive" assumption often being wrong, it works surprisingly well in practice — especially for text.

### What you'll learn
- The conditional independence assumption and why it's "naive"
- Gaussian Naive Bayes for numeric features
- Multinomial Naive Bayes for text classification
- Why NB works well even when the independence assumption is violated
- NB vs logistic regression — when to use each

### Dataset: SMS Spam (text classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

## 📐 Part 1 — Bayes with Independence Assumption

Full Bayes would require estimating P(X₁=x₁, X₂=x₂, ..., Xₚ=xₚ | Y=k) — the joint distribution of all features. With p=1000 features this is impossible.

**Naive Bayes** assumes conditional independence:
```
P(X₁,...,Xₚ | Y=k) = P(X₁|Y=k) × P(X₂|Y=k) × ... × P(Xₚ|Y=k)
```

Now we only need to estimate p separate univariate distributions — completely tractable.

The "naive" part: features are rarely truly independent. But the resulting classifier is surprisingly robust — the probability estimates may be wrong, but the *decision* (which class gets the highest probability) is often right.

In [ ]:
# Wine Quality dataset — predict whether a wine is high quality (score >= 7)
# Real sensory + chemical measurements from Portuguese wines
# Much more meaningful than Iris for business analytics students

import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.stats import norm
import matplotlib.pyplot as plt

try:
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    wine = pd.read_csv(url, sep=';')
    print("Loaded Wine Quality dataset from UCI")
except:
    from sklearn.datasets import load_wine
    data = load_wine(as_frame=True)
    wine = data.frame.rename(columns={'target': 'quality'})
    wine['quality'] = wine['quality'] * 3  # scale to 0-12 range
    print("Using sklearn wine dataset (fallback)")

# Binary target: quality >= 7 = "high quality"
wine['high_quality'] = (wine['quality'] >= 7).astype(int)
feature_cols = [c for c in wine.columns if c not in ['quality','high_quality']]
X_wine = wine[feature_cols]
y_wine = wine['high_quality']

print(f"Wine dataset: {wine.shape}")
print(f"High quality (score >= 7): {y_wine.mean():.1%}")
print(f"Features: {feature_cols}")
wine.head(3)


In [ ]:
# Gaussian NB on Wine Quality — show what it learns
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from scipy.stats import norm

X_wine_arr = X_wine.values
X_tr, X_te, y_tr, y_te = train_test_split(X_wine_arr, y_wine,
                                            test_size=0.25, random_state=42,
                                            stratify=y_wine)
gnb = GaussianNB()
gnb.fit(X_tr, y_tr)

print("=== Gaussian NB: Wine Quality ===")
print(classification_report(y_te, gnb.predict(X_te),
                            target_names=['Standard','High Quality']))

# Visualize: what distributions GNB learned for top features
top_features = ['alcohol', 'volatile acidity', 'sulphates']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, feat in zip(axes, top_features):
    feat_idx = list(X_wine.columns).index(feat)
    for cls, color, label in [(0,'#1e5fa8','Standard (<=6)'),
                               (1,'#e85d20','High Quality (>=7)')]:
        mu  = gnb.theta_[cls, feat_idx]
        std = np.sqrt(gnb.var_[cls, feat_idx])
        x_range = np.linspace(mu - 3.5*std, mu + 3.5*std, 200)
        ax.plot(x_range, norm.pdf(x_range, mu, std), color=color, lw=2, label=label)
        data_cls = X_wine.values[y_wine==cls, feat_idx]
        ax.hist(data_cls, bins=20, color=color, alpha=0.2, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle("Gaussian NB: Learned distributions per class\n(separation = predictive power)",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Alcohol content: high-quality wines cluster at higher values")
print("   Volatile acidity: high-quality wines have LOWER acidity (less vinegary)")
print("   Where the distributions overlap least = most discriminating feature")


In [ ]:
# Text classification: spam detection
X_text = sms['message'].values
y_text = sms['spam'].values

# Pipeline: TF-IDF → Multinomial NB
pipeline_mnb = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf',   MultinomialNB(alpha=1.0))  # alpha = Laplace smoothing
])

X_tr, X_te, y_tr, y_te = train_test_split(X_text, y_text, test_size=0.2, 
                                             random_state=42, stratify=y_text)
pipeline_mnb.fit(X_tr, y_tr)
y_pred = pipeline_mnb.predict(X_te)

print("=== Multinomial Naive Bayes: Spam Detection ===\n")
print(classification_report(y_te, y_pred, target_names=['Ham','Spam']))

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Ham','Spam']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Spam Detection')
plt.tight_layout()
plt.show()

In [ ]:
# Most discriminative words for spam vs ham
import re
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_vec = vectorizer.fit_transform(X_text)
mnb = MultinomialNB().fit(X_vec, y_text)

feature_names = vectorizer.get_feature_names_out()
# Log-ratio: how much more likely in spam vs ham
log_ratios = mnb.feature_log_prob_[1] - mnb.feature_log_prob_[0]

top_spam = pd.Series(log_ratios, index=feature_names).nlargest(15)
top_ham  = pd.Series(log_ratios, index=feature_names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(top_spam.index[::-1], top_spam.values[::-1], color='#e85d20', edgecolor='white')
axes[0].set_title('Top 15 Spam Words')
axes[0].set_xlabel('Log-ratio (spam/ham)')

axes[1].barh(top_ham.index, top_ham.abs().values, color='#1e5fa8', edgecolor='white')
axes[1].set_title('Top 15 Ham Words')
axes[1].set_xlabel('Log-ratio (ham/spam)')

plt.suptitle('Most Discriminative Words — Naive Bayes Spam Classifier', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 The model learned these patterns purely from labeled examples — no rules were written by hand")

In [ ]:
# @title 📝 Quiz — Naive Bayes
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the conditional independence assumption in Naive Bayes?
# @markdown **Q2:** What type of Naive Bayes would you use for word counts in documents?
# @markdown **Q3:** What is Laplace smoothing and why is it needed?
# @markdown **Q4:** Why does Naive Bayes sometimes outperform logistic regression on small datasets?
# @markdown **Q5:** What is one real-world scenario where the independence assumption is clearly violated but NB still works?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_="Naive Bayes"
# @title 🤖 AI Grading
# @markdown **Enter your GitHub username, then click ▶ Run.**
# @markdown Colab will send your answers to Gemini automatically — no key needed.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

import textwrap
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {v or '(no answer)'}"
                    for i,(_,v) in enumerate(answers.items(),1))

    prompt = (f"Grade these quiz answers for the \"{_NB_TITLE}\" notebook "
              f"(Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
              f"{qa}\n\n"
              f"For each answer say CORRECT, PARTIAL, or INCORRECT and one sentence why. "
              f"Accept any reasonable paraphrase as correct. "
              f"End with an overall grade (Excellent/Good/Needs Review/Incomplete) "
              f"and one sentence on what to review if they struggled.")

    # Try Colab's built-in Gemini via the generativeai SDK (no key needed in Colab)
    success = False
    try:
        import google.generativeai as genai
        # In Colab, calling generate_content without configure() uses
        # Colab's own managed credentials automatically
        model = genai.GenerativeModel("gemini-1.5-flash")
        resp  = model.generate_content(prompt)
        print("\n" + "\u2550"*56)
        print(f"  \U0001f916 Feedback \u2014 {_NB_TITLE}")
        if GITHUB_USERNAME.strip():
            print(f"  Student: @{GITHUB_USERNAME.strip()}")
        print("\u2550"*56)
        for line in resp.text.strip().split("\n"):
            if line.strip():
                for w in textwrap.wrap(line.strip(), 54):
                    print(f"  {w}")
            else:
                print()
        print("\u2550"*56)
        success = True
    except Exception:
        pass

    if not success:
        # Fallback: print the prompt so they can paste it into Colab's AI chat
        print("\u2550"*56)
        print("  Automatic grading unavailable.")
        print("  Paste the text below into the Gemini chat panel:")
        print("  (click the \u2728 sparkle icon in the Colab toolbar)")
        print("\u2550"*56)
        print()
        print(prompt)
        print()
        print("\u2550"*56)

    print("\n  \U0001f4be File \u2192 Save a copy in GitHub \u2192 your fork")


---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*